<a href="https://colab.research.google.com/github/quinn-dolan/portfolio/blob/main/assets/docs/combine_large_email_lists.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Standard library and third-party imports
import os
import pandas as pd

In [ ]:
# -------------------------------------------------------
# CONFIG: Update these values before running the notebook
# -------------------------------------------------------

# List of Excel files to process
file_list = ['file1.xlsx', 'file2.xlsx', 'file3.xlsx', 'file4.xlsx', 'file5.xlsx', 'file6.xlsx']

# Name of the output file where combined emails will be saved
output_filename = "combined_emails.csv"

In [ ]:
# Preview the column headers of each file without loading full data into memory
# Useful for confirming which email column name is used across files (e.g. 'email' vs 'email_address')
print(f"{'FILE NAME':<25} | {'COLUMNS FOUND'}")
print("-" * 60)

for file in file_list:
    try:
        # nrows=0 reads only the header row — no data loaded
        df_headers = pd.read_excel(file, nrows=0)
        columns = df_headers.columns.tolist()
        print(f"{file:<25} | {columns}")
    except Exception as e:
        # Catches missing files or unreadable Excel files
        print(f"{file:<25} | Error: {e}")

In [ ]:
# Accumulator list — each file's email DataFrame gets appended here
all_emails = []

for file in file_list:
    print(f"Processing {file}...")

    # Peek at headers only to identify the correct email column name
    df_headers = pd.read_excel(file, nrows=0)
    columns = df_headers.columns.tolist()

    # Check for known email column variants; add more elif branches as needed
    target_col = None
    if 'email' in columns:
        target_col = 'email'
    elif 'email_address' in columns:
        target_col = 'email_address'

    if target_col:
        # Load only the target column to minimize memory usage
        df = pd.read_excel(file, usecols=[target_col])

        # Standardize column name to 'email' for consistent concatenation later
        df.columns = ['email']

        # Drop blank rows and within-file duplicates before appending
        df = df.dropna(subset=['email']).drop_duplicates()
        all_emails.append(df)
        print(f"   Success: Extracted {len(df)} emails.")
    else:
        # File will be skipped — check its headers in Cell 3 to diagnose
        print(f"   Warning: No email column found in {file}!")

# Combine all individual DataFrames into one
print("Merging all files...")
final_df = pd.concat(all_emails, ignore_index=True)

# Remove duplicates that appear across multiple files
final_df = final_df.drop_duplicates()

# Write final deduplicated list to CSV
final_df.to_csv(output_filename, index=False)
print(f"Done! Saved {len(final_df)} unique emails to {output_filename}")